Imports and Setup

We'll use pandas for data manipulation and seaborn for clean, professional-grade visuals.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set the visual style
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

Data Loading & Preprocessing

This is the "heavy lifting" cell. It includes helper functions to convert 100.7K into 100700 and handle the free requests.

In [ ]:
def convert_tokens(value):
    """Converts token strings like '100.7K' or '1.2M' to actual integers."""
    if pd.isna(value) or value == "":
        return 0
    value = str(value).upper().strip()
    if 'K' in value:
        return int(float(value.replace('K', '')) * 1_000)
    if 'M' in value:
        return int(float(value.replace('M', '')) * 1_000_000)
    return int(float(value))

def convert_requests(value):
    """Handles numeric requests and 'free' labels."""
    val_str = str(value).lower().strip()
    if 'free' in val_str:
        return 1  # Counting a 'free' request as 1 request
    try:
        return int(float(val_str))
    except ValueError:
        return 0

# 1. Load your CSV
# Replace 'cursor_usage.csv' with your actual filename
df = pd.read_csv('cursor_usage.csv')

# 2. Clean the data
df['Date'] = pd.to_datetime(df['Date'])
df['Tokens_Num'] = df['Total Tokens'].apply(convert_tokens)
df['Requests_Num'] = df['Requests'].apply(convert_requests)

print(f"Data loaded: {len(df)} rows found.")
df.head()

Graphing Usage by Model

This cell creates a side-by-side comparison of Total Tokens and Total Requests for each model.

In [ ]:
# Aggregate data by Model
model_usage = df.groupby('Model').agg({
    'Tokens_Num': 'sum',
    'Requests_Num': 'sum'
}).sort_values(by='Tokens_Num', ascending=False).reset_index()

# Create the plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Total Tokens
sns.barplot(data=model_usage, x='Tokens_Num', y='Model', ax=ax1, palette="viridis")
ax1.set_title('Total Tokens Consumed by Model', fontweight='bold')
ax1.set_xlabel('Tokens')

# Plot 2: Total Requests
sns.barplot(data=model_usage, x='Requests_Num', y='Model', ax=ax2, palette="magma")
ax2.set_title('Total Requests by Model', fontweight='bold')
ax2.set_xlabel('Number of Requests')

plt.tight_layout()
plt.show()

Bonus - Timeline of Usage

Since you have a Date column, it’s often helpful to see when you are using these models most heavily.

In [ ]:
# Group by Date and Model
daily_usage = df.groupby([df['Date'].dt.date, 'Model'])['Tokens_Num'].sum().unstack().fillna(0)

daily_usage.plot(kind='area', stacked=True, figsize=(12, 6), alpha=0.7)
plt.title('Daily Token Usage by Model Over Time', fontsize=14, fontweight='bold')
plt.ylabel('Tokens')
plt.xlabel('Date')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()